# What is a Feature?

A feature is a numerical representation of a molecule that can be understood by machine learning algorithms.

Machine learning models cannot directly interpret molecular structures or SMILES strings.

Therefore, molecules must be converted into numerical features that describe their structural and physicochemical properties.

This process is called feature generation or feature engineering.

## Types of Molecular Features

Common molecular features include:

- Molecular descriptors
- Morgan fingerprints
- MACCS keys
- Physicochemical properties
- Topological descriptors
- Graph-based representations

# where are Features Used ?

# Types
# 1:Molecular Descriptors

Molecular descriptors are numerical values that describe the physicochemical, structural, or topological properties of a molecule.

Each descriptor represents a measurable property of the molecule.

Examples include:

- Molecular Weight
- LogP
- TPSA
- Number of Hydrogen Bond Donors
- Number of Hydrogen Bond Acceptors
- Number of Rotatable Bonds
- Number of Rings

# 2: Morgan Fingerprints

Morgan fingerprints represent the structural patterns present in a molecule.

Instead of measuring molecular properties, they encode the presence or absence of chemical substructures as binary values.

Each fingerprint bit corresponds to a specific structural environment within the molecule.

Typical fingerprint sizes are:

- 1024 bits
- 2048 bits

Bit1 = 0

Bit2 = 1

Bit3 = 0

Bit4 = 1

...

Bit2048 = 0

# 3:MACCS Keys

MACCS Keys are predefined structural fingerprints consisting of 166 bits.

Each bit corresponds to a specific predefined chemical pattern, such as:

- Aromatic ring
- Hydroxyl group
- Carboxylic acid
- Halogen atom

If the pattern is present, the bit is set to 1; otherwise, it is 0.

Hydroxyl Group

↓

Present

↓

Bit = 1

## Which Feature Type Should We Use?

There is no universally best feature representation.

The choice depends on the prediction task.

- Molecular descriptors are useful for capturing physicochemical properties.

- Morgan fingerprints are effective for representing molecular structure.

- MACCS Keys provide compact structural information.

In many QSAR applications, descriptors and fingerprints are combined to create a richer feature set.

In [1]:
# Practical Example
# Import Libraries

import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator

In [3]:
# Load ESOL Dataset

df = pd.read_csv("esol_raw.csv")

df = df[["smiles", "measured log solubility in mols per litre"]]

df["Mol"] = df["smiles"].apply(Chem.MolFromSmiles)

df = df[df["Mol"].notnull()]

print("Number of molecules:", len(df))

Number of molecules: 1128


In [4]:
# Generate Molecular Descriptor 

descriptor_data = []

for mol in df["Mol"]:

    descriptor_data.append([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.RingCount(mol),
        Descriptors.HeavyAtomCount(mol),
        Descriptors.FractionCSP3(mol),
        Descriptors.NumValenceElectrons(mol)
    ])

descriptor_df = pd.DataFrame(
    descriptor_data,
    columns=[
        "MolWt",
        "LogP",
        "TPSA",
        "HBD",
        "HBA",
        "RotatableBonds",
        "RingCount",
        "HeavyAtoms",
        "FractionCSP3",
        "ValenceElectrons"
    ]
)

descriptor_df.head()

,MolWt,LogP,TPSA,HBD,HBA,RotatableBonds,RingCount,HeavyAtoms,FractionCSP3,ValenceElectrons
0,457.432,-3.10802,202.32,7,12,7,3,32,0.650000,178
1,201.225,2.84032,42.24,1,2,2,2,15,0.083333,76
2,152.237,2.87800,17.07,0,1,4,0,11,0.500000,62
3,278.354,6.29940,0.00,0,0,0,5,22,0.000000,102
4,84.143,1.74810,0.00,0,1,0,1,5,0.000000,26


In [5]:
# Generate Morgan Fingerprints 

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

fingerprints = []

for mol in df["Mol"]:

    fp = morgan_gen.GetFingerprintAsNumPy(mol)

    fingerprints.append(fp)

fingerprint_df = pd.DataFrame(fingerprints)

fingerprint_df.head()

,0,1,2,3,4,5,6,7,8,9,...,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047
0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
# Comibine Both Features

X = pd.concat(
    [descriptor_df, fingerprint_df],
    axis=1
)

print("Feature Matrix Shape:", X.shape)

Feature Matrix Shape: (1128, 2058)


In [7]:
# Define the Target Variable

y = df["measured log solubility in mols per litre"]

print("Target Shape:", y.shape)

Target Shape: (1128,)


In [8]:
# Check the Final Dataset

print("Feature Matrix:", X.shape)
print("Target:", y.shape)

X.head()

Feature Matrix: (1128, 2058)
Target: (1128,)


,MolWt,LogP,TPSA,HBD,HBA,RotatableBonds,RingCount,HeavyAtoms,FractionCSP3,ValenceElectrons,...,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047
0,457.432,-3.10802,202.32,7,12,7,3,32,0.650000,178,...,0,0,0,0,0,0,0,0,0,0
1,201.225,2.84032,42.24,1,2,2,2,15,0.083333,76,...,0,0,0,0,0,0,0,0,0,0
2,152.237,2.87800,17.07,0,1,4,0,11,0.500000,62,...,0,0,0,0,0,0,0,0,0,0
3,278.354,6.29940,0.00,0,0,0,5,22,0.000000,102,...,0,0,0,0,0,0,0,0,0,0
4,84.143,1.74810,0.00,0,1,0,1,5,0.000000,26,...,0,0,0,0,0,0,0,0,0,0


In [9]:
print(X.head())

print("\nRows (Molecules):", X.shape[0])
print("Columns (Features):", X.shape[1])

print("\nDescriptor Columns:")
print(X.columns[:10])

print("\nFingerprint Columns:")
print(X.columns[10:20])

     MolWt     LogP    TPSA  HBD  HBA  RotatableBonds  RingCount  HeavyAtoms  \
0  457.432 -3.10802  202.32    7   12               7          3          32   
1  201.225  2.84032   42.24    1    2               2          2          15   
2  152.237  2.87800   17.07    0    1               4          0          11   
3  278.354  6.29940    0.00    0    0               0          5          22   
4   84.143  1.74810    0.00    0    1               0          1           5   

   FractionCSP3  ValenceElectrons  ...  2038  2039  2040  2041  2042  2043  \
0      0.650000               178  ...     0     0     0     0     0     0   
1      0.083333                76  ...     0     0     0     0     0     0   
2      0.500000                62  ...     0     0     0     0     0     0   
3      0.000000               102  ...     0     0     0     0     0     0   
4      0.000000                26  ...     0     0     0     0     0     0   

   2044  2045  2046  2047  
0     0     0     0   

## Interpretation

The ESOL dataset was transformed into a machine-learning-ready feature matrix.

For each molecule:

- 10 molecular descriptors were calculated to capture physicochemical properties.
- A 2048-bit Morgan fingerprint was generated to represent structural information.

These two feature sets were combined to create a final feature matrix containing 2058 features per molecule.

The target variable is the experimentally measured aqueous solubility.

This feature matrix serves as the input for QSAR and machine learning models.